In [ ]:
pip install xlrd>=2.0.1

In [ ]:
conda install -c anaconda xlrd

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [ ]:
def load_data():
    df = pd.read_excel("Sample - Superstore 2019.xls")
    return df
raw_df = load_data()
raw_df

In [ ]:
raw_df.columns

In [ ]:
raw_df.info() # memory usage: 1.6+ MB

In [ ]:
def clean_data(df):
    df = df.copy()
    print("Rows before cleaning:", len(df))

    # 1) Inconsistent formats: strip extra spaces from all text columns
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()

    # 2) Duplicate records
    dup_count = df.duplicated().sum()
    df = df.drop_duplicates()
    print("Duplicates removed:", dup_count)

    # 3) Missing values: numeric -> median, text -> mode
    print("Missing values before:\n", df.isnull().sum()[df.isnull().sum() > 0])
    for col in df.columns:
        if df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mode()[0])

    # 4) Outliers (IQR rule): flagged, NOT deleted, because very large orders
    #    in retail are real transactions and deleting them would bias Sales/Profit
    for col in ["Sales", "Profit"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"{col} Outlier"] = (df[col] < lower) | (df[col] > upper)
        print(f"{col}: {df[f'{col} Outlier'].sum()} outliers (IQR bounds: {lower:.1f} to {upper:.1f})")

    print("Rows after cleaning:", len(df))
    return df

In [ ]:
def transform_data(df):
    df = df.copy()
    # Postal Code is an identifier, so it should be text not a number
    df['Postal Code'] = df['Postal Code'].astype('Int64').astype('string')
    # Discount stays numeric (0.2) and we add a numeric percentage column (20)
    df['Discount %'] = df['Discount'] * 100
    return df

In [ ]:
def create_features(df):
    df = df.copy()
    # Profit Margin (%)
    df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
    # Shipping Duration (days)
    df['Shipping Duration'] = (df['Ship Date'] - df['Order Date']).dt.days
    # Sales Performance Category
    df['Sales Performance Category'] = pd.qcut(df['Sales'], q=3, labels=['Low', 'Medium', 'High'])
    # Year-Month based on ORDER date (the sale happens when the order is placed)
    df['Year Month'] = df['Order Date'].dt.to_period('M')
    return df

In [ ]:
def run_pipeline():
    """Runs the full preprocessing workflow in order: load -> clean -> transform -> features"""
    df = load_data()
    df = clean_data(df)
    df = transform_data(df)
    df = create_features(df)
    return df

df = run_pipeline()
df.dtypes

Statistical Analysis

In [ ]:
df[['Sales','Quantity','Discount','Profit','Profit Margin','Shipping Duration']].describe().round(2)

In [ ]:
# Skewness of numeric columns (|skew| > 1 means a strongly skewed distribution)
df[['Sales','Quantity','Discount','Profit','Profit Margin','Shipping Duration']].skew().round(2)

In [ ]:
# Correlation Matrix
corr = df[['Sales','Quantity','Discount','Profit','Profit Margin','Shipping Duration']].corr()
corr.round(2)

Insights & KPIs

In [ ]:
# Sales vs. Profit Relationship (KPIs 1)
df['Sales'].corr(df['Profit'])

In [ ]:
# Total Sales (KPIs 2)
df['Sales'].sum()

In [ ]:
# Total Profit (KPIs 3)
df['Profit'].sum()

In [ ]:
# Overall Profit Margin % (KPIs 4) = Total Profit / Total Sales
df['Profit'].sum() / df['Sales'].sum() * 100

In [ ]:
# Total Orders (KPIs 5)
df["Order ID"].nunique()

In [ ]:
# Total Customers (KPIs 6)
df["Customer ID"].nunique()

In [ ]:
# Average Order Value متوسط قيمة الطلب الواحد (KPIs 7)
order_sales = df.groupby("Order ID")["Sales"].sum()
order_sales.mean().round()

In [ ]:
# Average Shipping Duration (KPIs 8)
df['Shipping Duration'].mean().round()

In [ ]:
# Automated KPIs
def generate_kpis(df):
    order_sales = df.groupby("Order ID")["Sales"].sum()
    KPIs = {
        "Total Sales": df["Sales"].sum(),
        "Total Profit": df["Profit"].sum(),
        "Overall Profit Margin %": df["Profit"].sum() / df["Sales"].sum() * 100,
        "Total Orders": df["Order ID"].nunique(),
        "Total Customers": df["Customer ID"].nunique(),
        "Average Shipping Duration": df['Shipping Duration'].mean().round(),
        "Sales vs. Profit Relationship": df['Sales'].corr(df['Profit']),
        "Average Order Value": order_sales.mean().round(),
    }
    return KPIs
kpi_summary = generate_kpis(df)
# تحويل القاموس إلى DataFrame
kpi_df = pd.DataFrame(list(kpi_summary.items()), columns=["KPI", "Value"])
# حفظ كـ CSV
kpi_df.to_csv("KPI_summary.csv", index=False)
kpi_df

In [ ]:
# Top-Performing Product Categories (Insight 1)
sales_by_category = df.groupby('Category')['Sales'].sum().sort_values(ascending = False).round()
sales_by_category

In [ ]:
# Regional Sales Performance (Insight 2)
sales_by_region = df.groupby('Region')['Sales'].sum().sort_values(ascending = False).round()
sales_by_region

In [ ]:
# Best-Selling Products (Insight 3)
sales_by_subcategory = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending = False).round()
sales_by_subcategory

In [ ]:
# Most Profitable Products (Insight 4)
profit_by_subcategory = df.groupby('Sub-Category')['Profit'].sum().sort_values(ascending = False).round()
profit_by_subcategory

In [ ]:
# High-Sales, Low-Profit Products المنتجات ذات المبيعات المرتفعة والأرباح المنخفضة. (Insight 5)
insight = df.groupby('Sub-Category')[['Sales', 'Profit']].sum()
insight['Sales Category'] = pd.qcut(
    insight['Sales'],
    q=3,
    labels=['Low', 'Medium', 'High'])
insight['Profit Category'] = pd.qcut(
    insight['Profit'],
    q=3,
    labels=['Low', 'Medium', 'High'])
high_sales_low_profit = insight[
    (insight['Sales Category'] == 'High') &
    (insight['Profit Category'] == 'Low')]
high_sales_low_profit

In [ ]:
# Top 5 Customer Order Activity نعرف أكتر خمس عملاء طلبوا أوردرات من عندنا (Insight 6)
top_5_customers = df.groupby(['Customer ID','Customer Name'])['Order ID'].nunique().sort_values(ascending = False).head(5)
top_5_customers

In [ ]:
# Top 5 Customers by Sales (Insight 7)
top_5_customers_sales = df.groupby(['Customer ID','Customer Name'])['Sales'].sum().sort_values(ascending = False).head(5)
top_5_customers_sales

In [ ]:
# Regional Shipping Performance (Insight 8)
shipping_by_region = df.groupby('Region')['Shipping Duration'].mean().sort_values(ascending = True)
shipping_by_region

In [ ]:
# Trend Sales (Insight 9) - based on Order Date
monthly_sales = df.groupby('Year Month')['Sales'].sum()
monthly_sales

In [ ]:
# Trend Profit (Insight 10) - based on Order Date
monthly_profit = df.groupby('Year Month')['Profit'].sum()
monthly_profit

Charts (Visualization)

In [ ]:
sales_by_category.plot(kind="bar")
plt.title("Sales Performance by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.savefig("Sales Performance by Category.png")
plt.show()

In [ ]:
sales_by_region.plot(kind="bar")
plt.title("Regional Sales Performance")
plt.xlabel("Region")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.savefig("Regional Sales Performance.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    sales_by_subcategory.index,
    sales_by_subcategory.values)
plt.title("Best-Selling Sub-Categories")
plt.xlabel("Sub-Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("Best-Selling Sub-Categories.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    profit_by_subcategory.index,
    profit_by_subcategory.values)
plt.title("Most Profitable by Sub-Categories")
plt.xlabel("Sub-Category")
plt.ylabel("Total Profit")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("Most Profitable by Sub-Categories.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['Sales'], df['Profit'])
plt.title("Sales vs. Profit Relationship")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.tight_layout()
plt.savefig("Sales vs. Profit Relationship.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    shipping_by_region.index,
    shipping_by_region.values)
plt.title("Regional Shipping Performance")
plt.xlabel("Region")
plt.ylabel("Average Shipping Duration (Days)")
plt.savefig("Regional Shipping Performance.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    top_5_customers.index.get_level_values('Customer Name'),
    top_5_customers.values)
plt.title("Top 5 Customer by Order Activity")
plt.xlabel("Customer")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("Top 5 Customer by Order Activity.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    top_5_customers_sales.index.get_level_values('Customer Name'),
    top_5_customers_sales.values)
plt.title("Top 5 Customers by Sales")
plt.xlabel("Customer")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("Top 5 Customers by Sales.png")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(high_sales_low_profit))
plt.bar(
    [i - 0.2 for i in x],
    high_sales_low_profit['Sales'],
    width=0.4,
    label='Sales')
plt.bar(
    [i + 0.2 for i in x],
    high_sales_low_profit['Profit'],
    width=0.4,
    label='Profit')
plt.title("High-Sales, Low-Profit Sub-Categories")
plt.xlabel("Sub-Category")
plt.ylabel("Amount")
plt.xticks(
    x,
    high_sales_low_profit.index,
    rotation=45,
    ha='right')
plt.legend()
plt.tight_layout()

plt.savefig("High-Sales, Low-Profit Sub-Categories.png")
plt.show()

In [ ]:
# Trend Sales
plt.figure(figsize=(10,6))
monthly_sales.plot(kind='line', marker='o')
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.savefig("Monthly Sales Trend.png")
plt.show()

In [ ]:
# Trend Profit
plt.figure(figsize=(10,6))
monthly_profit.plot(kind='line', marker='o', color='green')
plt.title("Monthly Profit Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Profit")
plt.xticks(rotation=45)
plt.savefig("Monthly Profit Trend.png")
plt.show()

Seaborn Charts (Statistical Analysis)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("Correlation Heatmap.png")
plt.show()

In [ ]:
# Distribution of Profit (clipped to the 1st-99th percentile so extreme values don't hide the shape)
low, high = df['Profit'].quantile([0.01, 0.99])
plt.figure(figsize=(9, 5))
sns.histplot(df['Profit'].clip(low, high), bins=50, kde=True)
plt.title("Profit Distribution (1st-99th percentile)")
plt.xlabel("Profit")
plt.tight_layout()
plt.savefig("Profit Distribution.png")
plt.show()

In [ ]:
# Profit by Category (outliers visible in the boxplot)
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Category', y='Profit')
plt.title("Profit Distribution by Category")
plt.tight_layout()
plt.savefig("Profit Boxplot by Category.png")
plt.show()

In [ ]:
# Does a higher discount hurt the profit margin?
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Discount %', y='Profit Margin')
plt.title("Profit Margin by Discount Level")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("Profit Margin by Discount.png")
plt.show()

In [ ]:
# Sales Performance Category counts per Region
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x='Region', hue='Sales Performance Category')
plt.title("Sales Performance Category by Region")
plt.tight_layout()
plt.savefig("Sales Performance Category by Region.png")
plt.show()

In [ ]:
df.info() # memory usage: 1.8+ MB

In [ ]:
df.to_csv("cleaned_superstore.csv", index=False)